# 04 · Limpieza Avanzada — INGRESANTES

**Objetivo:** aplicar a `ingresantes_clean.parquet` todas las correcciones planificadas y generar `ingresantes_clean_v2.parquet`, dejando la capa Silver **inmaculada** antes de pasar a Gold.

**Entrada:** `data/Silver/ingresantes_clean.parquet` (~3.1 M filas, 1.4 GB)
**Salida:** `data/Silver/ingresantes_clean_v2.parquet`

**Dataset:** anual (granularidad persona × programa × año). Clave candidata del contrato: `CODIGO_INEI + GUID_PERSONA + CODIGO_SIU_PROGRAMA`. El diagnóstico (`03_diagnostico_silver.ipynb`) reportó **293 duplicados residuales** bajo esa clave.

**Correcciones a aplicar (en orden):**
1. Eliminar duplicados (293).
2. Imputación de grupos de carrera (Nivel 1 diccionario + Nivel 2 centinela).
3. Imputación de textos (`DEPARTAMENTO_NACIMIENTO`, `NACIONALIDAD`) → `"NO ESPECIFICADO"`.
4. Imputación de `ANIO_NACIMIENTO` con la mediana (entero).
5. Normalización de textos (strip + colapsar espacios internos).
6. Compactar discapacidad en `TIENE_DISCAPACIDAD` (bool) y eliminar las 7 columnas originales.
7. Guardar `ingresantes_clean_v2.parquet`.
8. Liberar memoria (`del ing; gc.collect()`).

**Nota:** este notebook procesa **únicamente Ingresantes** con **pandas** (dataset pequeño, rápido). **No se procesan Matriculados aquí.**

In [1]:
from pathlib import Path

# Detección automática de la raíz del proyecto (misma lógica que los notebooks 01–03)
current_dir = Path.cwd()
if (current_dir / "data").exists():
    PROJECT_ROOT = current_dir
elif (current_dir.parent / "data").exists():
    PROJECT_ROOT = current_dir.parent
else:
    raise FileNotFoundError(
        "No se encontró la carpeta 'data'. Ejecuta este notebook desde la raíz "
        "del proyecto o desde notebooks/."
    )

SILVER = PROJECT_ROOT / "data" / "Silver"
ING_PATH = SILVER / "ingresantes_clean.parquet"
ING_V2 = SILVER / "ingresantes_clean_v2.parquet"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Entrada existe:", ING_PATH.exists(), "→", ING_PATH)

PROJECT_ROOT: /mnt/datos/Proyectos/A.Prueba Tecnica UCSP
Entrada existe: True → /mnt/datos/Proyectos/A.Prueba Tecnica UCSP/data/Silver/ingresantes_clean.parquet


## Imports y utilidades

Se usan **pandas** (dataset pequeño, ~3.1 M filas) con `dtype_backend='pyarrow'` para mantener los datos compactos y reducir la memoria.

In [2]:
import gc
import os

import numpy as np
import pandas as pd

print("pandas", pd.__version__)


def rss_actual_gb():
    """RSS actual del proceso en GB (Linux, /proc/self/statm)."""
    try:
        with open("/proc/self/statm", encoding="utf-8") as fh:
            paginas = int(fh.read().split()[1])
        return paginas * os.sysconf("SC_PAGE_SIZE") / (1024**3)
    except (OSError, ValueError, IndexError):
        return float("nan")


print(f"RSS inicial: {rss_actual_gb():.2f} GB")

pandas 3.0.5
RSS inicial: 0.13 GB


## Carga del dataset

In [3]:
import sys
print(sys.executable)

/home/jcc/Proyectos/A.Prueba Tecnica UCSP/.venv/bin/python


In [4]:
import sys
print("Python:", sys.version)
import pandas as pd
print("pandas:", pd.__version__)
try:
    import pyarrow
    print("pyarrow:", pyarrow.__version__)
except ImportError:
    print("❌ pyarrow NO instalado")

Python: 3.12.13 (main, Apr  7 2026, 20:45:25) [Clang 22.1.1 ]
pandas: 3.0.5
pyarrow: 25.0.1


In [5]:
ing = pd.read_parquet(ING_PATH, engine="pyarrow", dtype_backend="pyarrow")

CLAVE_ING = ["CODIGO_INEI", "GUID_PERSONA", "CODIGO_SIU_PROGRAMA"]
DISC_COLS = [c for c in ing.columns if c.startswith("DES_DISCAPACIDAD")]

print(f"Shape inicial: {ing.shape[0]:,} filas × {ing.shape[1]} columnas")
print(f"Columnas de discapacidad ({len(DISC_COLS)}):", DISC_COLS)
print(f"RSS tras carga: {rss_actual_gb():.2f} GB")

Shape inicial: 3,120,287 filas × 33 columnas
Columnas de discapacidad (7): ['DES_DISCAPACIDAD_DE_COMUNICACION', 'DES_DISCAPACIDAD_DE_CONDUCTA', 'DES_DISCAPACIDAD_DE_DESTREZA', 'DES_DISCAPACIDAD_DE_DISPOSICION', 'DES_DISCAPACIDAD_DE_LOCOMOCION', 'DES_DISCAPACIDAD_DE_SITUACION', 'DES_DISCAPACIDAD_DEL_CUIDADO']
RSS tras carga: 2.03 GB


### 1. Eliminar duplicados

Bajo la clave `CODIGO_INEI + GUID_PERSONA + CODIGO_SIU_PROGRAMA`, conservando la primera ocurrencia. Se espera eliminar **293** duplicados.

In [6]:
filas_iniciales = len(ing)

ing.drop_duplicates(subset=CLAVE_ING, keep="first", inplace=True, ignore_index=True)

eliminados = filas_iniciales - len(ing)
print(f"Filas iniciales: {filas_iniciales:,}")
print(f"Filas después de deduplicar: {len(ing):,}")
print(f"Duplicados eliminados: {eliminados:,}")
assert eliminados == 293, f"Se esperaban 293 duplicados, se eliminaron {eliminados}"

Filas iniciales: 3,120,287
Filas después de deduplicar: 3,119,994
Duplicados eliminados: 293


### 2. Imputación de GRUPOS (doble nivel)

- **Nivel 1 (Diccionario de la verdad):** se construye `map_grupos` a partir de las filas **sin nulos** en `CODIGO_GRUPO_1` y `CODIGO_GRUPO_3`, usando `CODIGO_SIU_PROGRAMA` como clave. Con `map` (vectorizado) se rellenan los nulos de `CODIGO_GRUPO_1`, `NOMBRE_GRUPO_1`, `CODIGO_GRUPO_3`, `NOMBRE_GRUPO_3`.
- **Nivel 2 (Centinela):** lo que quede nulo tras el mapeo se rellena con `-1` (códigos) y `"SIN CLASIFICACION"` (nombres).

In [7]:
GRUPO_COLS = ["CODIGO_GRUPO_1", "NOMBRE_GRUPO_1", "CODIGO_GRUPO_3", "NOMBRE_GRUPO_3"]

# Nivel 1 · Construir el diccionario desde filas sin nulos en ambos códigos
mapeo_src = ing[
    ing["CODIGO_GRUPO_1"].notna() & ing["CODIGO_GRUPO_3"].notna()
][["CODIGO_SIU_PROGRAMA"] + GRUPO_COLS].drop_duplicates("CODIGO_SIU_PROGRAMA")

map_grupos = mapeo_src.set_index("CODIGO_SIU_PROGRAMA")[GRUPO_COLS].to_dict("index")
print(f"Mapeo construido para {len(map_grupos):,} programas de {ing['CODIGO_SIU_PROGRAMA'].nunique():,}")

# Nivel 1 · Rellenar nulos con el mapeo (vectorizado con map, rápido para ~3M filas)
for c in GRUPO_COLS:
    ing[c] = ing[c].mask(
        ing[c].isna(),
        ing["CODIGO_SIU_PROGRAMA"].map({k: v[c] for k, v in map_grupos.items()}),
    )

# Nivel 2 · Centinela para los residuales
ing["CODIGO_GRUPO_1"] = ing["CODIGO_GRUPO_1"].fillna(-1)
ing["CODIGO_GRUPO_3"] = ing["CODIGO_GRUPO_3"].fillna(-1)
ing["NOMBRE_GRUPO_1"] = ing["NOMBRE_GRUPO_1"].fillna("SIN CLASIFICACION")
ing["NOMBRE_GRUPO_3"] = ing["NOMBRE_GRUPO_3"].fillna("SIN CLASIFICACION")

print(f"Nulos restantes en grupos: {ing[GRUPO_COLS].isna().sum().sum()}")
print(f"Centinelas -1 (CODIGO_GRUPO_1): {(ing['CODIGO_GRUPO_1'] == -1).sum():,}")
print(f"Centinelas -1 (CODIGO_GRUPO_3): {(ing['CODIGO_GRUPO_3'] == -1).sum():,}")
print(f"'SIN CLASIFICACION' (NOMBRE_GRUPO_1): {(ing['NOMBRE_GRUPO_1'] == 'SIN CLASIFICACION').sum():,}")
print(f"'SIN CLASIFICACION' (NOMBRE_GRUPO_3): {(ing['NOMBRE_GRUPO_3'] == 'SIN CLASIFICACION').sum():,}")

Mapeo construido para 411 programas de 534
Nulos restantes en grupos: 0
Centinelas -1 (CODIGO_GRUPO_1): 11,008
Centinelas -1 (CODIGO_GRUPO_3): 11,008
'SIN CLASIFICACION' (NOMBRE_GRUPO_1): 0
'SIN CLASIFICACION' (NOMBRE_GRUPO_3): 0


### 3. Imputación de textos

`DEPARTAMENTO_NACIMIENTO` y `NACIONALIDAD` → `"NO ESPECIFICADO"`.

In [8]:
ing["DEPARTAMENTO_NACIMIENTO"] = ing["DEPARTAMENTO_NACIMIENTO"].fillna("NO ESPECIFICADO")
ing["NACIONALIDAD"] = ing["NACIONALIDAD"].fillna("NO ESPECIFICADO")
print(f"Nulos restantes DEPARTAMENTO_NACIMIENTO: {ing['DEPARTAMENTO_NACIMIENTO'].isna().sum()}")
print(f"Nulos restantes NACIONALIDAD: {ing['NACIONALIDAD'].isna().sum()}")

Nulos restantes DEPARTAMENTO_NACIMIENTO: 0
Nulos restantes NACIONALIDAD: 0


### 4. Imputación de `ANIO_NACIMIENTO`

Los nulos se rellenan con la **mediana** redondeada a entero.

In [9]:
mediana_anio = int(round(ing["ANIO_NACIMIENTO"].median()))
print(f"Mediana de ANIO_NACIMIENTO (entero): {mediana_anio}")
ing["ANIO_NACIMIENTO"] = ing["ANIO_NACIMIENTO"].fillna(mediana_anio).astype(int)
print(f"Nulos restantes ANIO_NACIMIENTO: {ing['ANIO_NACIMIENTO'].isna().sum()}")

Mediana de ANIO_NACIMIENTO (entero): 2002
Nulos restantes ANIO_NACIMIENTO: 0


### 5. Normalización de textos

Se normalizan **todas** las columnas de tipo string (`object`/`string`):
1. `str.strip()` → elimina espacios al inicio y final.
2. `str.replace(r'\s+', ' ', regex=True)` → colapsa múltiples espacios internos a uno solo.

Así se eliminan los espacios dobles y los espacios en bordes en todos los campos de texto.

In [10]:
# Identificar columnas de tipo string (object o string)
STRING_COLS = [c for c in ing.columns if pd.api.types.is_string_dtype(ing[c])]
print(f"Columnas string a normalizar ({len(STRING_COLS)}):", STRING_COLS)

# Aplicar strip + colapso de espacios internos a cada columna string
for c in STRING_COLS:
    ing[c] = ing[c].str.strip().str.replace(r"\s+", " ", regex=True)

print(f"Espacios en bordes restantes: {sum(ing[c].str.match(r'^ | $').sum() for c in STRING_COLS):,}")
print(f"Espacios dobles restantes: {sum(ing[c].str.contains(r'  ', regex=False).sum() for c in STRING_COLS):,}")

Columnas string a normalizar (20): ['CODIGO_INEI', 'NOMBRE_ENTIDAD', 'TIPO_ENTIDAD', 'TIPO_GESTION', 'LICENCIADO', 'TIPO_CONSTITUCION', 'NIVEL_ACADEMICO', 'GUID_PERSONA', 'SEXO', 'NACIONALIDAD', 'DEPARTAMENTO_NACIMIENTO', 'EDAD', 'NOMBRE_GRUPO_1', 'NOMBRE_GRUPO_3', 'NOMBRE_PROGRAMA', 'ES_SEDE_PRINCIPAL', 'CODIGO_SIU_FILIAL', 'DEPARTAMENTO_FILIAL', 'PROVINCIA_FILIAL', 'CERT_GRAVEDAD']
Espacios en bordes restantes: 0
Espacios dobles restantes: 0


### 6. Transformación de DISCAPACIDAD

Se crea `TIENE_DISCAPACIDAD = True` si **cualquiera** de las 7 columnas `DES_DISCAPACIDAD_*` tiene un valor no nulo, y se eliminan las 7 columnas originales.

In [11]:
ing["TIENE_DISCAPACIDAD"] = ing[DISC_COLS].notna().any(axis=1)
ing = ing.drop(columns=DISC_COLS)
print(f"Personas con discapacidad (TIENE_DISCAPACIDAD=True): {ing['TIENE_DISCAPACIDAD'].sum():,}")
print(f"Columnas de discapacidad eliminadas: {len(DISC_COLS)}")
print(f"Shape final: {ing.shape[0]:,} filas × {ing.shape[1]} columnas")

Personas con discapacidad (TIENE_DISCAPACIDAD=True): 13,381
Columnas de discapacidad eliminadas: 7
Shape final: 3,119,994 filas × 27 columnas


### 7. Guardado y Liberación

Se guarda `ingresantes_clean_v2.parquet` y se **libera la memoria** con `del ing; gc.collect()`.

In [12]:
ING_V2.parent.mkdir(parents=True, exist_ok=True)
ing.to_parquet(ING_V2, engine="pyarrow", index=False)
print(f"Guardado: {ING_V2}")
print(f"Tamaño en disco: {ING_V2.stat().st_size / 1e9:.2f} GB")

Guardado: /mnt/datos/Proyectos/A.Prueba Tecnica UCSP/data/Silver/ingresantes_clean_v2.parquet
Tamaño en disco: 0.14 GB


In [13]:
# Resumen de transformaciones
print("=" * 70)
print("RESUMEN DE TRANSFORMACIONES — INGRESANTES")
print("=" * 70)
print(f"Filas iniciales:                 {filas_iniciales:,}")
print(f"Duplicados eliminados:          {eliminados:,}  (esperado 293)")
print(f"Filas finales:                  {len(ing):,}")
print(f"Columnas finales:               {ing.shape[1]}")
print(f"Nulos imputados en grupos:      {ing[GRUPO_COLS].isna().sum().sum()} (0 restantes)")
print(f"Nulos imputados en texto:       DEPARTAMENTO_NACIMIENTO y NACIONALIDAD → 'NO ESPECIFICADO'")
print(f"Nulos imputados en ANIO_NACIMIENTO: mediana {mediana_anio}")
print(f"Discapacidad compactada:        {len(DISC_COLS)} columnas → TIENE_DISCAPACIDAD")
print("=" * 70)

# Verificación final de duplicados
dup_final = len(ing) - len(ing.drop_duplicates(subset=CLAVE_ING))
print(f"Duplicados bajo clave (debe ser 0): {dup_final}")
assert dup_final == 0
assert ing[GRUPO_COLS].isna().sum().sum() == 0
assert ing[["DEPARTAMENTO_NACIMIENTO", "NACIONALIDAD"]].isna().sum().sum() == 0
assert ing["ANIO_NACIMIENTO"].isna().sum() == 0
print("✅ Ingresantes limpio: 0 duplicados, 0 nulos en columnas imputadas.")

RESUMEN DE TRANSFORMACIONES — INGRESANTES
Filas iniciales:                 3,120,287
Duplicados eliminados:          293  (esperado 293)
Filas finales:                  3,119,994
Columnas finales:               27
Nulos imputados en grupos:      0 (0 restantes)
Nulos imputados en texto:       DEPARTAMENTO_NACIMIENTO y NACIONALIDAD → 'NO ESPECIFICADO'
Nulos imputados en ANIO_NACIMIENTO: mediana 2002
Discapacidad compactada:        7 columnas → TIENE_DISCAPACIDAD
Duplicados bajo clave (debe ser 0): 0
✅ Ingresantes limpio: 0 duplicados, 0 nulos en columnas imputadas.


In [14]:
# FIN · Liberación total
del ing
gc.collect()
print(f"Memoria RSS tras liberar Ingresantes: {rss_actual_gb():.2f} GB")
print("✅ Notebook completado. Solo se procesó Ingresantes.")

Memoria RSS tras liberar Ingresantes: 2.11 GB
✅ Notebook completado. Solo se procesó Ingresantes.
